
# Magic: comparação aberto vs fechado para $\sigma_x$, $\sigma_y$ e $\sigma_z$

Este notebook é focado **somente nos casos de magic**:

- `magic_gauss_epsilon`
- `magic_gauss_T`
- `magic_cos_omega`

Ele lê os resultados gerados por:

```bash
python expected_values_all_article_cases_with_xy.py
```

A ideia aqui é comparar **dois a dois**:

- `const_aberto` vs `const`
- `var_aberto_min` vs `var_min`
- `var_aberto_av` vs `var_av`
- `var_aberto_max` vs `var_max`

Para cada caso de magic, o notebook gera uma figura com linhas:

```text
linha 1: <sigma_x>
linha 2: <sigma_y>
linha 3: <sigma_z>

coluna 1: constante
coluna 2: min
coluna 3: av
coluna 4: max
```

Em cada painel, ficam sobrepostas as curvas **aberto** e **fechado**.

> Importante: para esta comparação funcionar, o script precisa ter sido rodado com `RUN_CLOSED_CASES = True`.


In [ ]:

import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (15, 8),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})



## Configuração

Se quiser usar uma pasta específica de resultados, coloque o caminho em `ROOT_EV`.
Se deixar `None`, ele pega automaticamente a rodada mais recente em:

```text
results/expected_values_all_cases_xy/expected_values_all_cases_xy*
```


In [ ]:

def latest_root(base, prefix):
    folders = sorted(glob.glob(os.path.join(base, f"{prefix}*")))
    if not folders:
        raise FileNotFoundError(
            f"Não encontrei pastas em {base}/{prefix}*. "
            "Rode primeiro: python expected_values_all_article_cases_with_xy.py"
        )
    return folders[-1]


ROOT_EV = None
if ROOT_EV is None:
    ROOT_EV = latest_root("results/expected_values_all_cases_xy", "expected_values_all_cases_xy")

print("ROOT_EV =", ROOT_EV)



## Funções auxiliares


In [ ]:

def ev_case_dir(case_id):
    return os.path.join(ROOT_EV, case_id)


def load_observables(case_id, label):
    path = os.path.join(ev_case_dir(case_id), f"{label}_observables.csv")
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    required = ["time", "expect_X", "expect_Y", "expect_Z"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(
            f"O arquivo {path} não tem as colunas {missing}. "
            "Rode novamente expected_values_all_article_cases_with_xy.py."
        )
    return df


def load_case_map():
    path = os.path.join(ROOT_EV, "case_map.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Não encontrei {path}")
    return pd.read_csv(path)


def get_magic_cases():
    case_map = load_case_map()

    if "resource" in case_map.columns:
        magic = case_map.loc[case_map["resource"].astype(str).str.lower() == "magic"].copy()
    else:
        magic = case_map.loc[case_map["case_id"].astype(str).str.contains("magic", case=False, regex=False)].copy()

    if len(magic) == 0:
        raise RuntimeError("Não encontrei casos de magic no case_map.csv")

    preferred_order = ["magic_gauss_epsilon", "magic_gauss_T", "magic_cos_omega"]
    magic["_order"] = magic["case_id"].apply(
        lambda x: preferred_order.index(x) if x in preferred_order else 999
    )
    return magic.sort_values(["_order", "case_id"]).drop(columns=["_order"])


PAIR_SPECS = [
    {"name": "constante", "open": "const_aberto", "closed": "const"},
    {"name": "min",       "open": "var_aberto_min", "closed": "var_min"},
    {"name": "av",        "open": "var_aberto_av",  "closed": "var_av"},
    {"name": "max",       "open": "var_aberto_max", "closed": "var_max"},
]

SIGMA_SPECS = [
    {"column": "expect_X", "label": r"$\langle \sigma_x \rangle$", "name": "sigma_x"},
    {"column": "expect_Y", "label": r"$\langle \sigma_y \rangle$", "name": "sigma_y"},
    {"column": "expect_Z", "label": r"$\langle \sigma_z \rangle$", "name": "sigma_z"},
]



## Casos de magic encontrados


In [ ]:

magic_cases = get_magic_cases()
magic_cases



## Função principal de plot

Cada figura compara aberto e fechado **no mesmo painel** para cada par correspondente.


In [ ]:

def nice_case_title(row):
    case_id = row["case_id"]
    if "scan_display_name" in row and pd.notna(row["scan_display_name"]):
        return f"{case_id} — {row['scan_display_name']}"
    if "scan_name" in row and pd.notna(row["scan_name"]):
        return f"{case_id} — {row['scan_name']}"
    return str(case_id)


def plot_magic_open_closed_sigma_pairs(case_id, title=None, save=False):
    nrows = len(SIGMA_SPECS)
    ncols = len(PAIR_SPECS)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(16, 8.5),
        sharex=True,
        sharey="row",
        constrained_layout=True,
    )

    found_any = False

    for col_idx, pair in enumerate(PAIR_SPECS):
        df_open = load_observables(case_id, pair["open"])
        df_closed = load_observables(case_id, pair["closed"])

        for row_idx, sigma in enumerate(SIGMA_SPECS):
            ax = axes[row_idx, col_idx]

            if df_open is not None:
                ax.plot(
                    df_open["time"],
                    df_open[sigma["column"]],
                    lw=2.0,
                    linestyle="-",
                    label="aberto",
                )
                found_any = True

            if df_closed is not None:
                ax.plot(
                    df_closed["time"],
                    df_closed[sigma["column"]],
                    lw=2.0,
                    linestyle="--",
                    label="fechado",
                )
                found_any = True

            if df_open is None and df_closed is None:
                ax.text(
                    0.5, 0.5,
                    "sem dados",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )

            if row_idx == 0:
                ax.set_title(pair["name"])

            if col_idx == 0:
                ax.set_ylabel(sigma["label"])

            if row_idx == nrows - 1:
                ax.set_xlabel("t")

            if row_idx == 0 and col_idx == ncols - 1:
                ax.legend(loc="best")

    if not found_any:
        raise RuntimeError(
            f"Não encontrei observáveis abertos/fechados para {case_id}. "
            "Verifique se o script foi rodado com RUN_CLOSED_CASES = True."
        )

    fig.suptitle(title if title is not None else case_id, fontsize=14)

    if save:
        out = os.path.join(ev_case_dir(case_id), f"plot_magic_open_closed_sigma_pairs_{case_id}.png")
        fig.savefig(out, dpi=220, bbox_inches="tight")
        print("salvo:", out)

    plt.show()



## Plotar um caso específico


In [ ]:

# Escolha um caso da tabela magic_cases
case_id = magic_cases.iloc[0]["case_id"]
case_id


In [ ]:

row = magic_cases.loc[magic_cases["case_id"] == case_id].iloc[0]
plot_magic_open_closed_sigma_pairs(case_id, title=nice_case_title(row), save=False)



## Plotar todos os casos de magic

Mude `SAVE_ALL = True` para salvar automaticamente as figuras dentro da pasta de cada caso.


In [ ]:

SAVE_ALL = False

for _, row in magic_cases.iterrows():
    cid = row["case_id"]
    plot_magic_open_closed_sigma_pairs(cid, title=nice_case_title(row), save=SAVE_ALL)



## Conferência rápida de arquivos esperados

Esta célula mostra quais pares aberto/fechado existem para cada caso de magic.


In [ ]:

rows = []
for cid in magic_cases["case_id"]:
    for pair in PAIR_SPECS:
        rows.append({
            "case_id": cid,
            "pair": pair["name"],
            "open_label": pair["open"],
            "open_exists": load_observables(cid, pair["open"]) is not None,
            "closed_label": pair["closed"],
            "closed_exists": load_observables(cid, pair["closed"]) is not None,
        })

pd.DataFrame(rows)
